In [29]:
!nvidia-smi
from numba import cuda
import numpy as np
@cuda.jit
def hello_kernel():
    # Thread and block indices
    tx = cuda.threadIdx.x
    bx = cuda.blockIdx.x
    bdim = cuda.blockDim.x

    # Global thread ID
    gid = bx * bdim + tx

    print("Hello from Block", bx, "Thread", tx, "Global ID", gid)
blocks = 2
threads_per_block = 4

hello_kernel[blocks, threads_per_block]()
cuda.synchronize()

Wed Feb 18 10:11:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   72C    P0             31W /   70W |     523MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:680: NumbaPerformanceWarning: Grid size 2 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [30]:
#1D MATRIX ADDITION
from numba import cuda
import numpy as np

@cuda.jit
def add_1d(a, b, c):
    i = cuda.grid(1)         
    if i < c.size:            
        c[i] = a[i] + b[i]
N = 10                       
a = np.arange(N, dtype=np.int32)
b = np.arange(N, dtype=np.int32) * 10
c = np.zeros(N, dtype=np.int32)

d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_c = cuda.to_device(c)

threads_per_block = 128
blocks = (N + threads_per_block - 1) // threads_per_block

add_1d[blocks, threads_per_block](d_a, d_b, d_c)
cuda.synchronize()

result = d_c.copy_to_host()

print("A:", a)
print("B:", b)
print("C = A + B:", result)

A: [0 1 2 3 4 5 6 7 8 9]
B: [ 0 10 20 30 40 50 60 70 80 90]
C = A + B: [ 0 11 22 33 44 55 66 77 88 99]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:680: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [31]:
#2D MATRIX ADDITION
from numba import cuda
import numpy as np

@cuda.jit
def add_2d(a, b, c):
    row, col = cuda.grid(2)    

    if row < c.shape[0] and col < c.shape[1]:
        c[row, col] = a[row, col] + b[row, col]

rows, cols = 4, 5

A = np.arange(rows * cols, dtype=np.int32).reshape(rows, cols)
B = np.ones((rows, cols), dtype=np.int32) * 10
C = np.zeros((rows, cols), dtype=np.int32)

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.to_device(C)

threads_per_block = (16, 16)
blocks_per_grid_x = (cols + threads_per_block[1] - 1) // threads_per_block[1]
blocks_per_grid_y = (rows + threads_per_block[0] - 1) // threads_per_block[0]
blocks = (blocks_per_grid_y, blocks_per_grid_x)


add_2d[blocks, threads_per_block](d_A, d_B, d_C)
cuda.synchronize()

result = d_C.copy_to_host()

print("Matrix A:\n", A)
print("Matrix B:\n", B)
print("Matrix C = A + B:\n", result)

Matrix A:
 [[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]
Matrix B:
 [[10 10 10 10 10]
 [10 10 10 10 10]
 [10 10 10 10 10]
 [10 10 10 10 10]]
Matrix C = A + B:
 [[10 11 12 13 14]
 [15 16 17 18 19]
 [20 21 22 23 24]
 [25 26 27 28 29]]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:680: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


In [32]:
#CUDA-Based Parallel Vector Addition Using Numba in Python
from numba import cuda
import numpy as np

@cuda.jit
def add_arrays(a, b, c):
    i = cuda.grid(1)             
    if i < c.size:              
        c[i] = a[i] + b[i]

n = 1024

a = np.arange(n, dtype=np.float32)
b = np.arange(n, dtype=np.float32)

c = np.zeros(n, dtype=np.float32)

d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_c = cuda.to_device(c)

threads_per_block = 256
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

add_arrays[blocks_per_grid, threads_per_block](d_a, d_b, d_c)

cuda.synchronize()

c = d_c.copy_to_host()

# Print result
print("First 10 results:", c[:10])

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:680: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


First 10 results: [ 0.  2.  4.  6.  8. 10. 12. 14. 16. 18.]


In [33]:
#2D Parallel Matrix Multiplication Using CUDA (Numba) in Python
from numba import cuda
import numpy as np
# CUDA Kernel
@cuda.jit
def matmul_2d(A, B, C):
    row, col = cuda.grid(2)

    if row < C.shape[0] and col < C.shape[1]:
        temp = 0.0
        for k in range(A.shape[1]):
            temp += A[row, k] * B[k, col]
        C[row, col] = temp

# Host Code
M, K, N = 4, 3, 5   # A: MxK, B: KxN, C: MxN

A = np.arange(M * K, dtype=np.float32).reshape(M, K)
B = np.arange(K * N, dtype=np.float32).reshape(K, N)
C = np.zeros((M, N), dtype=np.float32)

# Copy to GPU
d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.to_device(C)

# 2D CUDA configuration
threads_per_block = (16, 16)
blocks_per_grid_x = (N + threads_per_block[1] - 1) // threads_per_block[1]
blocks_per_grid_y = (M + threads_per_block[0] - 1) // threads_per_block[0]
blocks_per_grid = (blocks_per_grid_y, blocks_per_grid_x)

# Launch kernel
matmul_2d[blocks_per_grid, threads_per_block](d_A, d_B, d_C)
cuda.synchronize()


# Copy result back
C = d_C.copy_to_host()

print("Matrix A:\n", A)
print("Matrix B:\n", B)
print("Matrix C = A x B:\n", C)

/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:680: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))


Matrix A:
 [[ 0.  1.  2.]
 [ 3.  4.  5.]
 [ 6.  7.  8.]
 [ 9. 10. 11.]]
Matrix B:
 [[ 0.  1.  2.  3.  4.]
 [ 5.  6.  7.  8.  9.]
 [10. 11. 12. 13. 14.]]
Matrix C = A x B:
 [[ 25.  28.  31.  34.  37.]
 [ 70.  82.  94. 106. 118.]
 [115. 136. 157. 178. 199.]
 [160. 190. 220. 250. 280.]]


In [34]:
#RGB TO GREYSCALE and the time diff

import numpy as np
import torch
import time

# Create 4K synthetic image
H, W = 4096, 4096
img = np.random.randint(0, 256, (H, W, 3), dtype=np.uint8)

# CPU timing (NumPy)
start_cpu = time.time()
gray_cpu = 0.299*img[:,:,2] + 0.587*img[:,:,1] + 0.114*img[:,:,0]
end_cpu = time.time()

cpu_time = end_cpu - start_cpu
print("CPU Time:", cpu_time, "seconds")

# GPU timing (PyTorch)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

img_gpu = torch.tensor(img, dtype=torch.float32).to(device)

# make sure GPU is ready
if device.type == "cuda":
    torch.cuda.synchronize()

start_gpu = time.time()

gray_gpu = 0.299*img_gpu[:,:,2] + 0.587*img_gpu[:,:,1] + 0.114*img_gpu[:,:,0]

if device.type == "cuda":
    torch.cuda.synchronize()

end_gpu = time.time()

gpu_time = end_gpu - start_gpu
print("GPU Time:", gpu_time, "seconds")

# Time Difference
print("Time Difference (CPU - GPU):", cpu_time - gpu_time, "seconds")

CPU Time: 0.18668079376220703 seconds
Device: cuda
GPU Time: 0.005151271820068359 seconds
Time Difference (CPU - GPU): 0.18152952194213867 seconds


In [35]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))


CUDA available: True
Device: cuda


In [36]:
import os
print(os.listdir('/kaggle/input'))

['notebooks']


In [37]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    break


/kaggle/input


In [38]:
try:
    print("Starting test...")

    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using:", device)

    class MyCNN(nn.Module):
        def __init__(self):
            super(MyCNN, self).__init__()
            self.conv1 = nn.Conv2d(3, 16, 3, 1, 1)
            self.pool = nn.MaxPool2d(2, 2)
            self.fc1 = nn.LazyLinear(1)



        def forward(self, x):
            x = self.pool(F.relu(self.conv1(x)))
            x = torch.flatten(x, 1)
            x = torch.sigmoid(self.fc1(x))
            print(x.shape)

            return x

    model = MyCNN().to(device)
    print("Model created")

    x = torch.randn(1, 3, 128, 128).to(device)
    out = model(x)

    print("Output shape:", out.shape)

except Exception as e:
    print("Error:", e)

Starting test...
Using: cuda
Model created
torch.Size([1, 1])
Output shape: torch.Size([1, 1])
